# Thông tin sinh viên
### MSSV: 24520220
### Họ và tên: Võ Hoài Chiều


# Đồ án: Nhập Môn Thị Giác Máy Tính


### Khai báo thư viện

In [6]:
import os
import hashlib
import numpy as np
import cv2
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import itertools
import torch.optim as optim

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
from torchvision.datasets import ImageFolder



### Kết nối/ Mount đến google driver

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Đọc dữ liệu ảnh từ các thư mục

In [8]:
# Hàm đọc dữ liệu ảnh từ thư mục
def load_images_from_folder(folder_path):
    images = []
    labels = []

    class_names = sorted(os.listdir(folder_path))
    for label, class_name in enumerate(class_names):
        class_folder = os.path.join(folder_path, class_name)
        if os.path.isdir(class_folder):
            for filename in os.listdir(class_folder):
                img_path = os.path.join(class_folder, filename)
                img = cv2.imread(img_path)
                if img is not None:
                    images.append(img)
                    labels.append(label)

    return images, labels, class_names

In [9]:
# đọc dữ liệu từ thư mục Dataset
train_folder= "/content/drive/MyDrive/dataset/fruit/train"
test_folder= "/content/drive/MyDrive/dataset/fruit/test"
val_folder= "/content/drive/MyDrive/dataset/fruit/val"

train_images, train_lables, class_name =load_images_from_folder(train_folder)
test_images, test_lables , _ =load_images_from_folder(test_folder)
val_images, val_lables , _ =load_images_from_folder(val_folder)

y_train=np.array(train_lables)
y_test=np.array(test_lables)
y_val = np.array(val_lables)

In [10]:
print(f"Nhãn các class: {class_name}")
print(f"Tổng số ảnh trong tập train: {len(train_images)}")
print(f"Tổng số ảnh trong tập test : {len(test_images)}")
print(f"Tổng số ảnh trong tập validation : {len(val_images)}")

Nhãn các class: ['apple', 'banana', 'mango', 'orange', 'tomato']
Tổng số ảnh trong tập train: 4200
Tổng số ảnh trong tập test : 900
Tổng số ảnh trong tập validation : 900


In [11]:
print(train_images[0].shape)
print(train_lables[0])
print(y_train.shape)

(260, 260, 3)
0
(4200,)


## Tiền xử lí dữ liệu

### 1. Resize ảnh

In [12]:

# Định nghĩa phép biến đổi Resize riêng biệt
resize_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor() # Chuyển về Tensor [0, 1] để chuẩn bị cho bước chuẩn hóa
])

### 2. Chuẩn hóa pixel-normalization

In [13]:
# Thông số Mean và Std của ImageNet
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Định nghĩa phép chuẩn hóa riêng biệt
normalize_transform = transforms.Normalize(mean=imagenet_mean, std=imagenet_std)


### 3. Data Augmentation

In [14]:

# 1. Pipeline cho tập TRAIN (Có Augmentation để chống overfit)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Pipeline cho tập VAL/TEST
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### 4. Label Encoding

In [15]:

label_map = {
    'apple': 0,    # táo
    'banana': 1,   # chuối
    'mango': 2,   # xoài
    'orange': 3,    # cam
    'tomato': 4    # cà chua
}

train_ds = ImageFolder(root=train_folder, transform=train_transform) #Đây là cách nhanh nhất và phổ biến nhất để tải dữ liệu hình ảnh khi
val_ds   = ImageFolder(root=val_folder,   transform=test_transform) # cấu trúc thư mục đã được phân chia theo nhãn (label)
test_ds  = ImageFolder(root=test_folder,  transform=test_transform)

train_ds.classes = list(label_map.keys())
# 2. Cập nhật từ điển ánh xạ
train_ds.class_to_idx = label_map

# Lặp lại tương tự cho Val và Test
val_ds.classes = list(label_map.keys())
val_ds.class_to_idx = label_map
test_ds.classes = list(label_map.keys())
test_ds.class_to_idx = label_map

print(f"Thứ tự nhãn (Mapping): {train_ds.class_to_idx}")

train_loader = DataLoader(
                          train_ds,
                          batch_size=64,  # Sử dụng 2 luồng CPU để load data
                          shuffle=True,   # Tăng tốc độ chuyển dữ liệu từ CPU sang GPU
                          num_workers=2,
                          )
val_loader = DataLoader(
                        val_ds,
                        batch_size=64,
                        shuffle=False,
                        num_workers=2,

                        )

test_loader = DataLoader(
                        test_ds,
                        batch_size=64,
                        shuffle=False,
                        num_workers=2,

                        )



Thứ tự nhãn (Mapping): {'apple': 0, 'banana': 1, 'mango': 2, 'orange': 3, 'tomato': 4}


## Model ResNet18

### 1. Tranfer Learning

In [16]:

# 1. Tải mô hình ResNet18 với trọng số pretrained
model = models.resnet18(weights='IMAGENET1K_V1')

# 2. Kiểm tra số lượng input features của lớp Fully Connected (FC) cuối cùng
num_ftrs = model.fc.in_features

# 3. Thay thế lớp FC cuối cùng: 1000 lớp (ImageNet) -> 5 lớp (Trái cây)
model.fc = nn.Linear(num_ftrs, 5)

# Đưa mô hình lên GPU (nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# BIÊN DỊCH MÔ HÌNH (Mất khoảng vài phút ở epoch đầu nhưng các epoch sau sẽ nhanh hơn)
model = torch.compile(model)

# FINE-TUNE TOÀN BỘ
# Vì ảnh trái cây trong túi nilon có đặc thù rất khác ImageNet
for param in model.parameters():
    param.requires_grad = True

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


### 2.Loss Function

In [17]:

# 1. Loss Function: Sử dụng CrossEntropy cho bài toán phân loại nhiều lớp
criterion = nn.CrossEntropyLoss()


In [18]:
# In ra cấu hình lớp cuối cùng để xác nhận
print(model.fc)

# Tính tổng số tham số cần huấn luyện
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Tổng số tham số sẽ được cập nhật: {total_params:,}")

Linear(in_features=512, out_features=5, bias=True)
Tổng số tham số sẽ được cập nhật: 11,179,077


### 3. Early Stopping

In [19]:

class EarlyStopping:
    def __init__(self, patience=5, verbose=False, path='models/resnet18_fruit.pth'):
        self.patience = patience
        self.verbose = verbose #nếu true in log ra màn hình
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss # vì ta muốn loss nhỏ nhất là tốt nhưng logic lại so sánh kiểu 'score lớn là tốt' nên phải đảo dấu
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score:
            self.counter +=1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        if not os.path.exists('models'):
            os.makedirs('models')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

### 4. Train model and Validation

In [20]:


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, num_epochs=30, save_path='models/resnet18_fruit.pth'):
    since = time.time()
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    # 1. Khởi tạo Early Stopping (sử dụng class đã định nghĩa)
    early_stopping = EarlyStopping(patience=5, verbose=True, path=save_path)

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        # --- GIAI ĐOẠN TRAIN ---
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad() #Xóa các đạo hàm cũ,Nếu không xóa, các lỗi của đợt ảnh trước sẽ cộng dồn vào đợt sau, làm hỏng việc học.

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss.backward() # lan truyền ngược
            optimizer.step() # cập nhật trọng số

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)


        if scheduler is not None:
            scheduler.step()

        # --- GIAI ĐOẠN VALIDATION ---
        model.eval()
        val_running_loss = 0.0
        val_running_corrects = 0

        with torch.no_grad(): # không tính đạo hàm
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                val_running_loss += loss.item() * inputs.size(0)
                val_running_corrects += torch.sum(preds == labels.data)

        val_epoch_loss = val_running_loss / len(val_loader.dataset)
        val_epoch_acc = val_running_corrects.double() / len(val_loader.dataset)

        print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
        print(f'Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f}')

        # 2. Lưu lại lịch sử
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc.item())
        history['val_loss'].append(val_epoch_loss)
        history['val_acc'].append(val_epoch_acc.item())

        # 3. Gọi Early Stopping kiểm tra và lưu checkpoint
        early_stopping(val_epoch_loss, model)
        print("\n" + "-"*60 + "\n")

        if early_stopping.early_stop:
            print("Early stopping triggered. Kết thúc huấn luyện sớm.")
            break
        print()
    # 4. QUAN TRỌNG: Load lại trọng số tốt nhất trước khi trả về model
    # Tránh trường hợp model ở epoch cuối bị overfitting
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path))

    return model, history

### 5. GridSearch

In [21]:

# --- 1. KHỞI TẠO LẠI MÔ HÌNH ---
def reset_model():
    model_ft = models.resnet18(weights='IMAGENET1K_V1')
    model_ft.fc = nn.Linear(model_ft.fc.in_features, 5)
    model_ft = model_ft.to(device)
    return model_ft


# --- 2. CẤU HÌNH GRID SEARCH ---
param_grid = {
    'batch_size': [32,64,128],
    'lr': [0.001, 0.0001,0.00001],
    'gamma': [0.1,0.5]
}

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

best_val_acc = 0.0
best_params = None
summary_results = []




### 6. Train the model using GridSearch to find the optimal set of parameters

In [22]:
# --- 3. GIAI ĐOẠN 1: GRID SEARCH NHANH (3 EPOCHS) ---
print(">>> BẮT ĐẦU GRID SEARCH NHANH (3 EPOCH/TỔ HỢP)...")
for params in combinations:
    print(f"\nThử nghiệm: BS={params['batch_size']}, LR={params['lr']}, G={params['gamma']}")

    train_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True,num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=params['batch_size'], shuffle=False,num_workers=2)

    model_trial = reset_model()
    optimizer_trial = optim.Adam(model_trial.parameters(), lr=params['lr'], fused=True)

    # CHỈ TRAIN 3 EPOCH ĐỂ TÌM TÍN HIỆU
    _, history = train_model(model_trial, train_loader, val_loader, criterion, optimizer_trial, num_epochs=3)

    current_acc = history['val_acc'][-1]
    summary_results.append({**params, 'val_acc': current_acc})

    if current_acc > best_val_acc:
        best_val_acc = current_acc
        best_params = params
    del model_trial, optimizer_trial, train_loader, val_loader
    torch.cuda.empty_cache()

print(f"\n[KẾT QUẢ] Bộ tham số tối ưu nhất: {best_params}")

>>> BẮT ĐẦU GRID SEARCH NHANH (3 EPOCH/TỔ HỢP)...

Thử nghiệm: BS=32, LR=0.001, G=0.1
Epoch 1/3
----------
Train Loss: 0.3327 Acc: 0.8862
Val Loss: 0.2200 Acc: 0.9289
Validation loss decreased (inf --> 0.219966). Saving model...

------------------------------------------------------------


Epoch 2/3
----------
Train Loss: 0.1693 Acc: 0.9440
Val Loss: 0.1805 Acc: 0.9378
Validation loss decreased (0.219966 --> 0.180502). Saving model...

------------------------------------------------------------


Epoch 3/3
----------
Train Loss: 0.1528 Acc: 0.9526
Val Loss: 0.3192 Acc: 0.8922
EarlyStopping counter: 1 out of 5

------------------------------------------------------------



Thử nghiệm: BS=32, LR=0.001, G=0.5
Epoch 1/3
----------
Train Loss: 0.3193 Acc: 0.8938
Val Loss: 1.8230 Acc: 0.6856
Validation loss decreased (inf --> 1.823029). Saving model...

------------------------------------------------------------


Epoch 2/3
----------
Train Loss: 0.1892 Acc: 0.9433
Val Loss: 0.7173 Acc:

### Lưu mô hình

In [23]:
import torch
import os

# Tạo thư mục 'models' nếu nó chưa tồn tại
if not os.path.exists('models'):
    os.makedirs('models')

# Đường dẫn để lưu model
model_save_path = "/content/drive/MyDrive/CS231.Q21/models/resnet18_best.pth"

# Lưu state_dict của model
torch.save(final_model.state_dict(), model_save_path)
print(f"Mô hình đã được lưu tại: {model_save_path}")

NameError: name 'final_model' is not defined

### 7. Visualization analyze the impact of parameter on accuracy

In [ ]:
# --- 4.1 VẼ ĐỒ THỊ PHÂN TÍCH TỪNG THAM SỐ VỚI ACC ---
df_res = pd.DataFrame(summary_results)
plt.figure(figsize=(15, 4))
for i, col in enumerate(['batch_size', 'lr', 'gamma']):
    plt.subplot(1, 3, i+1)
    sns.barplot(x=col, y='val_acc', data=df_res)
    plt.title(f'Tác động của {col}')
plt.tight_layout()
plt.show()


In [ ]:
# --- 4.2 VẼ ĐỒ THỊ PHÂN TÍCH TỔ HỢP CÁC THAM SỐ VỚI ACC---
plt.figure(figsize=(16, 6))
unique_bs = df_res['batch_size'].unique()
for i, bs in enumerate(unique_bs):
    plt.subplot(1, len(unique_bs), i+1)
    subset = df_res[df_res['batch_size'] == bs]
    pivot_table = subset.pivot(index='lr', columns='gamma', values='val_acc')
    # Vẽ heatmap
    sns.heatmap(pivot_table, annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={'label': 'Validation Accuracy'})
    plt.title(f'Tổ hợp LR & Gamma (Batch Size = {bs})')
    plt.xlabel('Gamma')
    plt.ylabel('Learning Rate')

plt.suptitle('Sự tác động của tổ hợp 3 tham số lên độ chính xác', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

### 9. Train model with set of optimal parameter

In [ ]:
# --- 5. GIAI ĐOẠN 2: HUẤN LUYỆN VỚI CÁC THAM SỐ TỐI ƯU ---
print(f"\n>>> HUẤN LUYỆN CHÍNH THỨC VỚI BỘ THAM SỐ TỐI ƯU: {best_params}")

# Cấu hình lại với Best Params
train_loader = DataLoader(train_ds, batch_size=best_params['batch_size'], shuffle=True,num_workers=2)
val_loader = DataLoader(val_ds, batch_size=best_params['batch_size'], shuffle=False,num_workers=2)

final_model = reset_model()
optimizer_final = optim.Adam(final_model.parameters(), lr=best_params['lr'], fused=True)
# Thêm scheduler
scheduler_final = optim.lr_scheduler.StepLR(optimizer_final, step_size=7, gamma=best_params['gamma'])

# Huấn luyện kỹ (20 epochs + Early Stopping)
final_model, final_history = train_model(
    final_model, train_loader, val_loader, criterion, optimizer_final, num_epochs=20
)

In [ ]:

# --- 6. ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST ---
# 1. Dự đoán trên tập Test (Sử dụng final_model đã được load trọng số tốt nhất)
final_model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = final_model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())



### 10. Model Evaluation & Performance Metrics

In [ ]:
# 2. Xuất Accuracy, Precision, Recall, F1-score
# Sử dụng train_ds.classes để tự động lấy tên lớp, tránh gõ nhầm tên
target_names = train_ds.classes
print("\n" + "="*20 + " BÁO CÁO CHI TIẾT " + "="*20)
print(classification_report(y_true, y_pred, target_names=target_names))

# 3. Vẽ biểu đồ Loss/Accuracy theo epoch (Sử dụng final_history từ lần train cuối)
acc = final_history['train_acc']
val_acc = final_history['val_acc']
loss = final_history['train_loss']
val_loss = final_history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, label='Train Loss', color='blue', marker='o')
plt.plot(epochs_range, val_loss, label='Val Loss', color='orange', marker='s')
plt.title('Loss per Epoch (Final Model)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc, label='Train Acc', color='blue', marker='o')
plt.plot(epochs_range, val_acc, label='Val Acc', color='orange', marker='s')
plt.title('Accuracy per Epoch (Final Model)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# 4. Vẽ Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.title('Confusion Matrix - ResNet18 Optimized')
plt.show()